# AI Model Arena - Compare LLM Responses Side by Side

A Gradio-powered tool that sends the same prompt to GPT, Claude, and Gemini via OpenRouter and displays their responses side by side.

**Week 2 Community Contribution by Mirack**

Skills demonstrated: Multi-model comparison, Gradio UI, streaming, conversation management.

In [ ]:
import os
from openai import OpenAI
import gradio as gr

In [ ]:

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENAI_API_KEY")
)

# OpenRouter model names
GPT_MODEL = "openai/gpt-4.1-nano"
CLAUDE_MODEL = "anthropic/claude-sonnet-4-5"
GEMINI_MODEL = "google/gemini-2.0-flash-lite-001"

In [ ]:
def stream_model(prompt, history, model):
    messages = [{"role": "system", "content": "You are a helpful assistant. Keep responses concise."}]
    for human, ai in history:
        messages.append({"role": "user", "content": human})
        messages.append({"role": "assistant", "content": ai})
    messages.append({"role": "user", "content": prompt})

    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        if not chunk.choices:          
            continue                    
        content = chunk.choices[0].delta.content
        if content:
            result += content
            yield result

In [ ]:
test_prompt = "Explain what an API is in 2 sentences."
for name, model in [("GPT", GPT_MODEL), ("Claude", CLAUDE_MODEL), ("Gemini", GEMINI_MODEL)]:
    print(f"\n=== {name} ===")
    result = ""
    for chunk in stream_model(test_prompt, [], model):
        result = chunk
    print(result)

In [ ]:
def arena_compare(prompt, gpt_history, claude_history, gemini_history):
    """Send the same prompt to all 3 models and collect results."""
    results = {}
    for name, model, history in [
        ("gpt", GPT_MODEL, gpt_history),
        ("claude", CLAUDE_MODEL, claude_history),
        ("gemini", GEMINI_MODEL, gemini_history)
    ]:
        try:
            result = ""
            for chunk in stream_model(prompt, history, model):
                result = chunk
            results[name] = result
        except Exception as e:
            results[name] = f"Error: {e}"

    gpt_history.append((prompt, results["gpt"]))
    claude_history.append((prompt, results["claude"]))
    gemini_history.append((prompt, results["gemini"]))

    return (
        results["gpt"], results["claude"], results["gemini"],
        gpt_history, claude_history, gemini_history
    )

In [ ]:
with gr.Blocks(title="AI Model Arena") as arena:
    gr.Markdown("# AI Model Arena\nCompare GPT, Claude, and Gemini responses side by side — all via OpenRouter.")

    prompt_input = gr.Textbox(label="Your Prompt", placeholder="Ask anything...")
    submit_btn = gr.Button("Battle!", variant="primary")

    with gr.Row():
        gpt_output = gr.Textbox(label="GPT-4.1-nano", lines=10)
        claude_output = gr.Textbox(label="Claude Sonnet 4.5", lines=10)
        gemini_output = gr.Textbox(label="Gemini 2.0 Flash Lite", lines=10)

    gpt_state = gr.State([])
    claude_state = gr.State([])
    gemini_state = gr.State([])

    submit_btn.click(
        fn=arena_compare,
        inputs=[prompt_input, gpt_state, claude_state, gemini_state],
        outputs=[gpt_output, claude_output, gemini_output, gpt_state, claude_state, gemini_state]
    )

arena.launch()